In [43]:
import pandas as pd
import json
import random
import re
import csv
from pathlib import Path
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

random.seed(18)

# ----------------------------------
# 1. Templates
# ----------------------------------

from v03.data.dataset_templates import AFFIRMED_TEMPLATES, NEGATED_TEMPLATES,DISTRACTOR_TEMPLATES 


# ----------------------------------
# 2. Load symptom dictionary
# ----------------------------------

df = pd.read_csv(PROJECT_ROOT / "base_symptom_dict.csv")
df.head(2)

,path,level,prefLabel,synonym,definition,word_count_in_prefLabel,id
0,symptom,0,symptom,[],"A symptom is a perceived change in function, s...",1,s0001
1,symptom/musculoskeletal system symptom,1,musculoskeletal system symptom,[],NaN,3,s0002


In [44]:
df[df["id"] == "s0712"]

,path,level,prefLabel,synonym,definition,word_count_in_prefLabel,id
711,symptom/head and neck symptom/head symptom/mou...,4,dry mouth,[],NaN,2,s0712


# Generate Synthetic Samples

In [45]:

# ----------------------------------
# 3. Generate synthetic samples
# ----------------------------------

samples = []

for _, row in df.iterrows():
    symptom_id = row["id"]
    symptom_text = row["prefLabel"]

    # POS examples
    for tmpl in AFFIRMED_TEMPLATES:
        text = tmpl.format(SYMPTOM=symptom_text)
        samples.append({
            "text": text,
            "symptom_id": symptom_id,
            "is_negated": False
        })

    # NEG examples
    for tmpl in NEGATED_TEMPLATES:
        text = tmpl.format(SYMPTOM=symptom_text)
        samples.append({
            "text": text,
            "symptom_id": symptom_id,
            "is_negated": True
        })

    # Distraction template 
    for tmpl in DISTRACTOR_TEMPLATES:
        text = tmpl.format(SYMPTOM=symptom_text)
        samples.append({
            "text": text,
            "symptom_id": "",
            "is_negated": None
        })



In [46]:
# ----------------------------------
# 4. Shuffle
# ----------------------------------

random.shuffle(samples)


# ----------------------------------
# 5. Save to JSONL
# ----------------------------------

RAW_DATA_PATH = PROJECT_ROOT / "v03/data/synthetic_data.jsonl"
TOKENIZED_DATA_PATH = PROJECT_ROOT / "v03/data/synthetic_data_tokenized.jsonl"

def save_jsonl(filename, data):
    with open(filename, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

save_jsonl(RAW_DATA_PATH, samples)

print("Saved:", len(samples))
print("Raw data path:", RAW_DATA_PATH)

Saved: 106267
Raw data path: /Users/robertagarcia/Desktop/learning/bert_symptom_ner/v03/data/synthetic_data.jsonl


# White Space Tokenization

In [47]:
import pandas as pd
import json
from tqdm import tqdm
# Load the saved datasets back in
def load_jsonl(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(RAW_DATA_PATH)
print(f"Length of trianing data: {len(train_data)}")


Length of trianing data: 106267


In [48]:
TOKEN_PATTERN = re.compile(
    r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[^\sA-Za-z0-9]"
)

# ----------------------------
# Helpers
# ----------------------------
# Tokenizer that separates words and punctuation into separate tokens.
# Pattern: words with internal apostrophes or hyphens, or any single non-space punctuation char
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[^\sA-Za-z0-9]")

def tokenize_with_spans(text):
    """
    Return list of (token, start_char, end_char) using TOKEN_PATTERN.
    Example: "stridor." -> [("stridor", idx, idx+7), (".", idx+7, idx+8)]
    """
    tokens = []
    for m in TOKEN_PATTERN.finditer(text):
        tok = m.group(0)
        tokens.append((tok, m.start(), m.end()))
    return tokens

def normalize_token(tok):
    """Lowercase normalization for matching (leave punctuation tokens as-is)."""
    return tok.lower()

def find_subsequence(token_norms, target_tokens):
    """
    Find first index i where token_norms[i:i+len(target_tokens)] == target_tokens.
    Returns index or None.
    """
    n = len(target_tokens)
    if n == 0:
        return None
    for i in range(len(token_norms) - n + 1):
        ok = True
        for j in range(n):
            if token_norms[i + j] != target_tokens[j]:
                ok = False
                # Don't check the remaining tokens, if first one does not match, we cant match the remaining ones
                break
        if ok:
            return i
    return None

def symptom_to_tokenlist(symptom_text):
    """Convert symptom prefLabel to normalized token list (split on whitespace)."""
    # keep internal hyphens/apostrophes as part of tokens
    parts = [p for p in re.split(r"\s+", symptom_text.strip()) if p]
    parts_norm = [p.lower() for p in parts]
    return parts_norm


# ----------------------------
# Load symptom dictionary (id -> prefLabel)
# ----------------------------
symptom_map = {}
with open(f"{PROJECT_ROOT}/base_symptom_dict.csv", newline='', encoding='utf-8') as f:
    # assume CSV has header and column 'id' and 'prefLabel'
    reader = csv.DictReader(f)
    for r in reader:
        sid = r.get("id") or r.get("symptom_id") or r.get("ID")
        pref = r.get("prefLabel") or r.get("pref_label") or r.get("preflabel")
        if sid is None or pref is None:
            continue
        symptom_map[sid] = pref


In [49]:
PROJECT_ROOT

PosixPath('/Users/robertagarcia/Desktop/learning/bert_symptom_ner')

In [50]:
# ----------------------------
# Process input JSONL
# ----------------------------
not_found = []
samples = load_jsonl(RAW_DATA_PATH)


In [51]:
samples[0]

{'text': 'Clinical history reveals mammary gland inflammation.',
 'symptom_id': 's0286',
 'is_negated': False}

In [52]:
null_exs = [x for x in samples if x.get("is_negated") is None]
pos_exs = [x for x in samples if x.get("is_negated") is False]
neg_exs = [x for x in samples if x.get("is_negated") is True]

In [53]:
mini_batch = [
    null_exs[0],
    # pos_exs[0],
    # neg_exs[0]
]
mini_batch

[{'text': 'Screening form includes assessment of ventricular tachycardia.',
  'symptom_id': '',
  'is_negated': None}]

In [54]:
def process_sample(obj, not_found_log):
    text = obj.get("text", "")
    sid = obj.get("symptom_id") or ""
    is_negated = obj.get("is_negated", None)

    tokens_with_spans = tokenize_with_spans(text)
    tokens = [t for (t, s, e) in tokens_with_spans]
    token_norms = [normalize_token(t) for t in tokens]
    labels = ["O"] * len(tokens)

    if sid == "":
        out_obj = {
            "text": text,
            "word_tokens": tokens,
            "word_labels": labels,
            "symptom_id": "",
            "is_negated": None,
        }
        return out_obj, tokens_with_spans

    symptom_text = symptom_map.get(sid)
    if symptom_text is None:
        not_found_log.append({"text": text, "symptom_id": sid, "reason": "unknown_symptom_id"})
        out_obj = {
            "text": text,
            "word_tokens": tokens,
            "word_labels": labels,
            "symptom_id": sid,
            "is_negated": is_negated,
        }
        return out_obj, tokens_with_spans

    symptom_tokens = symptom_to_tokenlist(symptom_text)
    start_idx = find_subsequence(token_norms, symptom_tokens)

    if start_idx is None:
        stripped = [re.sub(r'^\W+|\W+$', '', t).lower() for t in tokens]
        start_idx = find_subsequence(stripped, symptom_tokens)

    if start_idx is not None:
        suffix = "NEG" if is_negated is True else "POS"
        b_label = f"B-SYMPTOM_{suffix}" # f"B-SYMPTOM_{sid}_{suffix}"
        i_label = f"I-SYMPTOM_{suffix}" #  f"I-SYMPTOM_{sid}_{suffix}"
        for k in range(len(symptom_tokens)):
            pos = start_idx + k
            if 0 <= pos < len(labels):
                labels[pos] = b_label if k == 0 else i_label
    else:
        not_found_log.append({"text": text, "symptom_id": sid, "symptom_text": symptom_text})

    out_obj = {
        "text": text,
        "word_tokens": tokens,
        "word_labels": labels,
        "symptom_id": sid,
        "is_negated": is_negated,
    }
    return out_obj, tokens_with_spans

# Side check: keep a mini-batch around for debugging before running the full export.
for obj in mini_batch:
    out_obj, tokens_with_spans = process_sample(obj, not_found)
    print(f"text: {obj.get('text', '')}")
    print(f"tokens_with_spans: {tokens_with_spans}")
    print(f"tokens: {out_obj['word_tokens']}")
    print(f"token_norms: {[normalize_token(t) for t in out_obj['word_tokens']]}")
    if out_obj["symptom_id"] == "":
        print("No symptom label detected!")
    else:
        print(f"symptom_text: {symptom_map.get(out_obj['symptom_id'])}")
        print(f"sid: {out_obj['symptom_id']}")
        print(f"is_neg: {out_obj['is_negated']}")
    print(f"out_obj: {out_obj}")
    print(f"len(word_tokens): {len(out_obj['word_tokens'])}, len(word_labels): {len(out_obj['word_labels'])}")


text: Screening form includes assessment of ventricular tachycardia.
tokens_with_spans: [('Screening', 0, 9), ('form', 10, 14), ('includes', 15, 23), ('assessment', 24, 34), ('of', 35, 37), ('ventricular', 38, 49), ('tachycardia', 50, 61), ('.', 61, 62)]
tokens: ['Screening', 'form', 'includes', 'assessment', 'of', 'ventricular', 'tachycardia', '.']
token_norms: ['screening', 'form', 'includes', 'assessment', 'of', 'ventricular', 'tachycardia', '.']
No symptom label detected!
out_obj: {'text': 'Screening form includes assessment of ventricular tachycardia.', 'word_tokens': ['Screening', 'form', 'includes', 'assessment', 'of', 'ventricular', 'tachycardia', '.'], 'word_labels': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], 'symptom_id': '', 'is_negated': None}
len(word_tokens): 8, len(word_labels): 8


In [55]:
# Full export: process every sample and write tokenized JSONL.
not_found = []

with open(TOKENIZED_DATA_PATH, "w", encoding="utf-8") as out_f:
    for obj in tqdm(samples):
        out_obj, _ = process_sample(obj, not_found)
        assert len(out_obj["word_tokens"]) == len(out_obj["word_labels"])
        out_f.write(json.dumps(out_obj, ensure_ascii=False) + "\n")

print("Saved tokenized samples to:", TOKENIZED_DATA_PATH)
print("Total not found / ambiguous matches:", len(not_found))
if len(not_found) > 0:
    print("Examples of not-found:")
    for e in not_found[:10]:
        print(e)


100%|██████████| 106267/106267 [00:00<00:00, 125457.14it/s]

Saved tokenized samples to: /Users/robertagarcia/Desktop/learning/bert_symptom_ner/v03/data/synthetic_data_tokenized.jsonl
Total not found / ambiguous matches: 0
